# Scientific Data Visualization
## 00-10 Review: from tables to questions
A table stores values; a chart helps us see comparisons, distributions and relationships. In Session 08 we selected numeric columns, kept identifiers and checked missing values. Today we ask **how do fictional gene measurements vary among four labels?** We first choose a chart, then write plotting commands.

Biology first: a gene is a DNA region involved in producing a functional product. Our gene labels and expression-like measurements are invented. Values use arbitrary units (AU), not RNA-seq counts or real experimental expression. We have one value per label, no replicates and no uncertainty model. These charts teach programming; they cannot establish significant differential expression or gene function.

All data are [synthetic educational tables](../../datasets/session-09/README.md), included for offline use. A blank means missing, not zero. A sequence length counts bases on one represented strand; GC percentage is (G+C)/length*100 for nonempty A/C/G/T DNA. Our tables have feasible chosen compositions but do not contain actual DNA strings.

Before class: use the environment from Session 08 with updated course requirements. Select the matching notebook kernel. No install command belongs in a Python cell. Enlarge text and work one cell at a time. Restart Kernel then Run All checks that no earlier hidden variables are needed.

### Reopen the small table
`pd.read_csv` loads a CSV; `shape`, `dtypes` and `isna().sum()` review its structure and missing values. The path cell accepts the course root or this notebook folder as the working directory. If the path fails, open the whole student-course folder and inspect `Path.cwd()`. `Path.cwd()` is the current working directory, not necessarily where a file is saved.

Expected: shape(4,2), gene IDs A-D, values8,12,4,10; no missing values. Checkpoint: which column is metadata? Why is zero different from blank?

In [ ]:
from pathlib import Path
import pandas as pd
course_root = Path.cwd()
if not (course_root / "datasets").is_dir():
    course_root = Path.cwd().parents[1]
data_folder = course_root / "datasets" / "session-09"
genes = pd.read_csv(data_folder / "measurements.csv")
print(genes)
print("Shape:", genes.shape)
print(genes.dtypes)
print("Missing values:", genes.isna().sum().to_dict())

## 10-25 Choose the chart before the command
Our question compares one invented measurement for each named gene. Gene ID is **categorical**: it names a category. Measurement is **quantitative**: numerical magnitude matters. A bar chart compares these categories with bar heights. The x axis runs horizontally; y runs vertically. Here x identifies the gene and y gives the measured value in AU.

An independent variable is a quantity chosen or used as an explanatory input; a dependent variable is the response measured. In a time study, time is often x and cell count y. In our category chart, x is a label, not a numeric cause. In a scatter chart, placing a variable on x does not prove it controls y. We only have fictional paired values.

| Question | Chart | x axis | y axis |
| --- | --- | --- | --- |
| How do four labels compare? | Bar | Gene ID (category) | Measurement (AU) |
| How are sequence lengths distributed? | Histogram | Length (bases, numeric bins) | Number of sequences |
| How do counts change at ordered times? | Line | Time (h) | Cell count (cells) |
| How are length and GC paired? | Scatter | Length (bases) | GC content (%) |

Use readable text, clear contrast, a descriptive title and honest scales. Avoid 3D effects or decorative clutter. A legend identifies distinct series when needed; do not add one merely to repeat the title. **Task 1:** choose and label each chart before seeing code. Do not connect unordered gene IDs with a line.

## 25-50 First Figure and Axes
Problem: draw the four-category comparison. Input: four gene IDs and four AU values in matching rows. Output: a labeled chart and a saved PNG. Algorithm: create an image and plotting area; draw bars; label; save; display; close.

`import matplotlib.pyplot as plt` imports the pyplot module from the Matplotlib package and gives it the conventional short name `plt`. A **Figure** is the entire image. An **Axes** is one plotting area inside it, containing both x and y axes. `plt.subplots()` creates a Figure and an Axes, returned as two values. `fig, ax = ...` assigns the first returned item to `fig` and the second to `ax`, from left to right. This is called unpacking; the two names match the two returned items. We use library objects without needing to define classes.

`figsize=(7,4)` gives width and height in inches. Larger figures help screen sharing. First create them and inspect the object types. We keep this Figure for the next cells.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4))
print(type(fig), type(ax))

`ax.bar(x,y)` draws one bar per category, with heights from matching numeric rows. `color='navy'` makes a dark high-contrast color. `set_xlabel`, `set_ylabel` and `set_title` add meaning; units belong in the axis label. `set_ylim(bottom=0)` starts bars at zero so the visible lengths represent magnitudes fairly. No legend is needed for one clearly named measurement series.

Predict Gene_B's height12 AU and Gene_C's height4 AU before running. Task 2: build this one command at a time; choose your own clear title.

In [ ]:
ax.bar(genes["gene_id"], genes["measurement_au"], color="navy")
ax.set_xlabel("Fictional gene identifier")
ax.set_ylabel("Measurement (AU)")
ax.set_title("Synthetic gene measurements")
ax.set_ylim(bottom=0)

### Save and display are different
`mkdir(parents=True, exist_ok=True)` creates the output folder and any missing parent folders; rerunning accepts an existing folder. `fig.savefig` writes this Figure to disk. The .png suffix selects PNG, a portable raster image format. `dpi=150` means150 pixels per inch; a7x4-inch Figure is approximately1050x600 pixels before cropping. `bbox_inches='tight'` includes the labels and can change exact dimensions. Increase dpi for a sharper export; it does not change the data.

`plt.show()` requests on-screen display; it does not save a file. A notebook often also displays a figure automatically at a cell boundary. Because we built one Figure over several cells, use `display(fig)` to explicitly render its finished state here. `display` is supplied by the notebook environment; scripts use `plt.show()` for an interactive window instead. Save before display/close. `plt.close(fig)` releases the Figure when finished, avoiding accumulated windows. `exists()` checks whether a path exists.

These original generated charts use the [included tables and their provenance](../../datasets/session-09/README.md). The exact named PNG is replaced on rerun; do not use that filename for unrelated work.

In [ ]:
output_folder = course_root / "outputs" / "session-09" / "lesson"
output_folder.mkdir(parents=True, exist_ok=True)
figure_path = output_folder / "gene_bar.png"
fig.savefig(figure_path, dpi=150, bbox_inches="tight")
print("PNG exists:", figure_path.exists())
display(fig)
plt.show()
plt.close(fig)

## 50-70 A bar chart is not a histogram
The bar chart used named categories. A histogram groups quantitative values into numeric intervals called **bins**. Its y axis counts observations, not GC percentages or gene values. Bins share a continuous numeric scale and adjacent intervals touch. Changing bin widths can change the apparent shape, so report the edges.

Problem: describe the distribution of eight fictional sequence lengths. Input:4,6,8,10,12,14,16,18 bases. Algorithm: group them into [0,5), [5,10), [10,15), [15,20]; count each group. Intervals include their left edge; their right edge is excluded except the final bin includes it. Expected counts[1,2,3,2], total8. The boundary10 belongs in the third bin. No frequency density or statistical inference is required.

`ax.hist` draws the histogram and returns counts, bin edges and bar shapes. We unpack these to inspect counts; we do not need to edit the shapes. `edgecolor` makes bins visible. `set_xticks` and `set_yticks` set explicit tick positions; integer y ticks avoid suggesting fractional sequence records. Task 3: predict wider-bin counts before trying [0,10,20].

In [ ]:
sequences = pd.read_csv(data_folder / "sequence_summary.csv")
print(sequences)
fig, ax = plt.subplots(figsize=(7, 4))
counts, edges, patches = ax.hist(
    sequences["length_bases"], bins=[0, 5, 10, 15, 20],
    color="teal", edgecolor="black"
)
ax.set_xlabel("Sequence length (bases)")
ax.set_ylabel("Number of sequences")
ax.set_title("Synthetic sequence length distribution")
ax.set_xticks([0, 5, 10, 15, 20])
ax.set_yticks([0, 1, 2, 3])
print("Bin counts:", counts.tolist())
fig.savefig(output_folder / "length_histogram.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

## 70-85 Ordered lines and paired points
A cell count is the number of cells at a time; here no volume or growth model is supplied. A line can show descriptive change across ordered times. `ax.plot(x,y,marker='o')` draws connecting segments and observed-point markers. The rows must be in time order. `label` names the series; `ax.legend()` shows that name.

The synthetic file has one blank count at hour4. Pandas reads it as NaN. Passing the complete column to Matplotlib leaves a gap there, not a point at zero. Do not silently replace the blank with zero or remove its row and draw a line across the gap. Neither a missing-point estimate nor a biological growth mechanism follows from this chart. Expected observed pairs:(0,10),(2,16),(6,28),(8,32). Task 4: check missing counts and predict the gap.

In [ ]:
time_series = pd.read_csv(data_folder / "time_series.csv")
print(time_series)
print("Missing values:", time_series.isna().sum().to_dict())
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(time_series["time_h"], time_series["cell_count"],
        marker="o", color="navy", label="Observed synthetic counts")
ax.set_xlabel("Time (h)")
ax.set_ylabel("Cell count (cells)")
ax.set_title("Synthetic cell counts; hour 4 is missing")
ax.set_xticks([0, 2, 4, 6, 8])
ax.set_ylim(bottom=0)
ax.legend()
fig.savefig(output_folder / "time_line.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

A scatter plot uses one point per pair of quantitative values. `ax.scatter(x,y)` does not connect points. Here each row provides one sequence length and its GC percentage. Keep both columns in the same row order; sorting x alone breaks the pairs. `set_ylim(-5,105)` adds a small display margin so markers at 0% and 100% are fully visible; it does not add values outside the valid percentage range. `set_yticks([0,25,50,75,100])` labels that valid range clearly.

Expected:S01=(4 bases,0%), S04=(10 bases,60%), S08=(18 bases,100%). The selected table has some upward pattern but not a perfect rule: S04 has60% at10 bases, while S05 has50% at12 bases. Association alone does not establish causality, and synthetic values are not biological evidence. Task 5: locate the three expected points and write one limitation.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(sequences["length_bases"], sequences["gc_percent"], color="teal")
ax.set_xlabel("Sequence length (bases)")
ax.set_ylabel("GC content (%)")
ax.set_title("Synthetic sequence length and GC content")
ax.set_ylim(-5, 105)
ax.set_yticks([0, 25, 50, 75, 100])
fig.savefig(output_folder / "length_gc_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

## 85-110 Practical: a Pandas-to-Matplotlib workflow
Use [Task 6](04_EXERCISES.md). Work from a fresh CSV read. State the question, inspect the rows/columns and missing values, choose the matching columns, create a new Figure/Axes for each chart, plot, label, compare values with the table, save, display and interpret.

First5 minutes: discuss the workflow and run [02_DEMO.py](02_DEMO.py), which saves the four taught charts without opening windows. The save-only scripts use `matplotlib.use("Agg")` before importing pyplot: Agg draws image files without a GUI window, so no desktop display is needed. This notebook uses inline display instead; do not copy the script backend setting into notebook cells. The script resolves inputs relative to its own file and writes under course outputs/session-09/demo. `plt.close` is useful for noninteractive scripts; the notebook uses show for inline displays. Open the PNG files in VS Code.

Next5 minutes guided: select sequence_id and gc_percent; plan a bar chart with y=0-100%. Next10 minutes independently: finish the bar and a length-versus-GC scatter; save practical_gc.png and practical_scatter.png. Final5 minutes compare/repair. Expected shape(8,3); no blanks; GC heights[0,50,25,60,50,50,75,100]; eight scatter points. Add your own code cells below this paragraph. State one descriptive result and one unsupported claim for each. Commands are already taught; the complete answer stays in the separate solutions.

## 110-120 Refine and assess
Review for3 minutes: compare each chart with its question. Are the axes meaningful and units correct? Is text readable at screen-sharing size? Are labels or legends useful? Does the scale support honest comparison? A bar baseline of9 would make values10 and12 look like heights1 and3, exaggerating their relative difference. Keep bars at zero. Scatter axes can use other justified ranges, but state them clearly. Missing measurements must not become zeros. Decoration cannot repair a wrong data-to-axis mapping.

Take the [four-minute assessment](07_ASSESSMENT.md), then spend3 minutes reviewing and reading [homework](05_HOMEWORK.md). Homework asks for three plots from a new small synthetic table, with reasons and limits for each.

Common errors: ModuleNotFoundError means the kernel may differ from the installed environment; reselect it. NameError means a prior cell did not run; restart and run in order. FileNotFoundError means check the course folder and cwd. Unequal x/y lengths mean check column/row selection. A blank saved image means save the specific Figure before showing/closing. Histograms of IDs confuse categories and numeric bins. Scientific claims need evidence beyond our invented values.

Offline fallback: use the CSVs and expected results above, sketch labeled charts, record the exact blocker and retry later. No microphone, webcam, external dataset download or slide deck is needed.